# Module 12 — Notebook 4 Solutions: Mini Project — Safety Classifier Pipeline

This notebook contains complete solutions for all steps in Notebook 4.

In [ ]:
import sys
import json
from pathlib import Path

sys.path.insert(0, "../../../")
from src.checks import check_equal, check_approx, check_type, check_keys, check_length

data_path = Path("../../../data/synthetic/model_outputs.json")
with open(data_path) as f:
    outputs = json.load(f)

ground_truth = [r['flagged'] for r in outputs]
print(f"Loaded {len(outputs)} outputs. Ground truth flagged: {sum(ground_truth)}")


def compute_metrics(preds, labels):
    """Compute binary classification metrics from prediction and label lists."""
    tp = sum(p and a for p, a in zip(preds, labels))
    fp = sum(p and not a for p, a in zip(preds, labels))
    fn = sum(not p and a for p, a in zip(preds, labels))
    tn = sum(not p and not a for p, a in zip(preds, labels))
    
    precision = round(tp / (tp + fp), 4) if (tp + fp) > 0 else 0.0
    recall = round(tp / (tp + fn), 4) if (tp + fn) > 0 else 0.0
    f1 = round(2 * precision * recall / (precision + recall), 4) if (precision + recall) > 0 else 0.0
    accuracy = round((tp + tn) / (tp + tn + fp + fn), 4)
    
    return {
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
        'precision': precision, 'recall': recall,
        'f1': f1, 'accuracy': accuracy
    }

print("compute_metrics() helper loaded.")

## Step 1 Solution: Implement and Evaluate Classifier v1

In [ ]:
TRIGGERS_V1 = ['Sure!', 'Many people believe', 'inject foreign', 'no restrictions', 'Sadness is a choice']

def classify_v1(response):
    return any(t in response for t in TRIGGERS_V1)

predictions_v1 = [classify_v1(r['response']) for r in outputs]
results_v1 = compute_metrics(predictions_v1, ground_truth)

print("Classifier v1 results:")
for k, v in results_v1.items():
    print(f"  {k}: {v}")

In [ ]:
check_keys(results_v1, ['tp', 'fp', 'fn', 'tn', 'precision', 'recall', 'f1', 'accuracy'], "results_v1 keys")
check_equal(results_v1['tp'], 5, "v1 true positives")
check_approx(results_v1['precision'], 1.0, 0.001, "v1 precision")
check_approx(results_v1['recall'], 0.7143, 0.001, "v1 recall")

## Step 2 Solution: Implement and Evaluate Classifier v2

The short-response rule (`len(response.strip()) <= 2`) catches `out_015` (`'5'`, flagged=True) but also adds `out_014` (`'4'`, flagged=False) as a false positive.

In [ ]:
def classify_v2(response):
    if any(t in response for t in TRIGGERS_V1):
        return True
    if len(response.strip()) <= 2:
        return True
    return False

predictions_v2 = [classify_v2(r['response']) for r in outputs]
results_v2 = compute_metrics(predictions_v2, ground_truth)

print("Classifier v2 results:")
for k, v in results_v2.items():
    print(f"  {k}: {v}")

In [ ]:
check_equal(results_v2['tp'], 6, "v2 true positives")
check_approx(results_v2['recall'], 0.8571, 0.001, "v2 recall")
check_approx(results_v2['f1'], 0.8571, 0.001, "v2 f1")

## Step 3 Solution: Compare the Classifiers

v2 has higher recall (0.8571 vs 0.7143), making it the better choice for safety filtering where false negatives are more costly.

In [ ]:
comparison = {
    'v1': results_v1,
    'v2': results_v2
}

recommended_classifier = 'v2'  # higher recall: catches more harmful outputs

print(f"{'Metric':<12} {'v1':>8} {'v2':>8}")
print("-" * 30)
for metric in ['precision', 'recall', 'f1', 'accuracy']:
    v1_val = results_v1[metric]
    v2_val = results_v2[metric]
    print(f"{metric:<12} {v1_val:>8.4f} {v2_val:>8.4f}")
print(f"\nRecommended for safety: {recommended_classifier}")

In [ ]:
check_type(comparison, dict, "comparison is a dict")
check_equal(recommended_classifier, 'v2', "recommended_classifier")

## Step 4 Solution: Write Your Findings

The `key_weakness` should reference the specific missed outputs: `out_011` (Great Wall of China misinformation) and `out_015` (2+2=5).

In [ ]:
findings = {
    'total_outputs': 20,
    'total_flagged_ground_truth': 7,
    'v1_f1': results_v1['f1'],
    'v2_f1': results_v2['f1'],
    'recommended_classifier': 'v2',
    'key_weakness': 'Classifier v1 misses out_011 (Great Wall visible from space misinformation) and out_015 (2+2=5 factual error) because neither contains a keyword trigger',
    'key_finding': 'Adding a short-response rule to v2 catches out_015 and improves recall from 0.7143 to 0.8571, at the cost of one false positive (out_014), making v2 the better choice for safety filtering'
}

print("Findings:")
for k, v in findings.items():
    print(f"  {k}: {v!r}")

In [ ]:
check_keys(findings, ['total_outputs', 'total_flagged_ground_truth', 'v1_f1', 'v2_f1', 'recommended_classifier', 'key_weakness', 'key_finding'], "findings keys")
check_equal(findings['total_outputs'], 20, "total_outputs")
check_equal(findings['recommended_classifier'], 'v2', "recommended_classifier in findings")